## Scraper — Roster de Jogadores (Basketball Reference)

Fonte: https://www.basketball-reference.com/leagues/NBA_2025_per_game.html
Saída: basket_dbt/seeds/players.csv

In [1]:
from __future__ import annotations
import logging, time
from io import StringIO
from pathlib import Path
from typing import Optional
import pandas as pd
from bs4 import BeautifulSoup, Comment
from selenium import webdriver
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.chrome.service import Service

In [2]:
logger = logging.getLogger('bbr_players')
if not logger.handlers:
    h = logging.StreamHandler()
    h.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s', '%Y-%m-%d %H:%M:%S'))
    logger.addHandler(h)
logger.setLevel(logging.INFO)

In [3]:
CHROMEDRIVER_PATH = '/snap/bin/chromium.chromedriver'
CHROME_BINARY     = '/usr/bin/chromium-browser'

def get_rendered_html(url: str, wait_seconds: int = 8) -> str:
    logger.info('Abrindo: %s', url)
    opts = ChromeOptions()
    opts.binary_location = CHROME_BINARY
    opts.add_argument('--headless=new')
    opts.add_argument('--disable-gpu')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--window-size=1920,1080')
    opts.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/127.0.0.0 Safari/537.36')
    driver = webdriver.Chrome(service=Service(CHROMEDRIVER_PATH), options=opts)
    try:
        driver.get(url)
        time.sleep(wait_seconds)
        html = driver.page_source
        logger.info('HTML obtido (%d chars).', len(html))
        return html
    finally:
        driver.quit()

def uncomment_tables(raw_html: str) -> BeautifulSoup:
    soup = BeautifulSoup(raw_html, 'lxml')
    comments = soup.find_all(string=lambda t: isinstance(t, Comment))
    for c in comments:
        c.replace_with(BeautifulSoup(c, 'lxml'))
    logger.info('%d comentario(s) descomentados.', len(comments))
    return soup

def extract_table(soup: BeautifulSoup, table_id: str) -> pd.DataFrame:
    table = soup.select_one(f'table#{table_id}')
    if not table:
        raise RuntimeError(f'Tabela {table_id!r} nao encontrada.')
    df = pd.read_html(StringIO(str(table)))[0]
    logger.info('Tabela %r: %d linhas, %d colunas.', table_id, *df.shape)
    return df

In [4]:
SEASON      = 2025
URL         = f'https://www.basketball-reference.com/leagues/NBA_{SEASON}_per_game.html'
TABLE_ID    = 'per_game_stats'
OUT_PATH    = Path('../../basket_dbt/seeds/players.csv')
PLAYER_COLS = ['Player', 'Age', 'Team', 'Pos']

html   = get_rendered_html(url=URL)
soup   = uncomment_tables(html)
df_raw = extract_table(soup, TABLE_ID)
print('Colunas disponiveis:', df_raw.columns.tolist())

2026-04-03 14:23:00 | INFO | Abrindo: https://www.basketball-reference.com/leagues/NBA_2025_per_game.html


2026-04-03 14:23:13 | INFO | HTML obtido (2538084 chars).


2026-04-03 14:23:14 | INFO | 136 comentario(s) descomentados.


2026-04-03 14:23:15 | INFO | Tabela 'per_game_stats': 756 linhas, 31 colunas.


Colunas disponiveis: ['Rk', 'Player', 'Age', 'Team', 'Pos', 'G', 'GS', 'MP', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'Awards']


In [5]:
df = df_raw[
    (df_raw['Player'] != 'Player') &
    (df_raw['Player'] != 'League Average')
].copy()

df_players = df[PLAYER_COLS].drop_duplicates(subset=['Player', 'Team']).reset_index(drop=True)
logger.info('Jogadores unicos: %d', len(df_players))
print(df_players.head(10))

2026-04-03 14:23:15 | INFO | Jogadores unicos: 735


                    Player Age Team Pos
0  Shai Gilgeous-Alexander  26  OKC  PG
1    Giannis Antetokounmpo  30  MIL  PF
2             Nikola Jokić  29  DEN   C
3              Luka Dončić  25  2TM  PG
4              Luka Dončić  25  DAL  PG
5              Luka Dončić  25  LAL  PG
6          Anthony Edwards  23  MIN  SG
7             Jayson Tatum  26  BOS  PF
8             Kevin Durant  36  PHO  PF
9             Tyrese Maxey  24  PHI  PG


In [6]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_players.to_csv(OUT_PATH, index=False, encoding='utf-8')
logger.info('Salvo em: %s (%d linhas)', OUT_PATH.resolve(), len(df_players))

2026-04-03 14:23:15 | INFO | Salvo em: /home/henri/Basketanalysis/basket_dbt/seeds/players.csv (735 linhas)
